In [1]:
import numpy as np
from pypdf import PdfReader
from groq import Groq

In [2]:
client = Groq(
    api_key="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"
)

In [3]:
reader = PdfReader("D:\Sahana\Project_RAG\Report.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

In [4]:
def chunk_text(text, chunk_size=300):

    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks


documents = chunk_text(text)

In [5]:
def create_embedding(text, dim=128):

    vector = np.zeros(dim)

    for word in text.lower().split():
        index = hash(word) % dim
        vector[index] += 1

    norm = np.linalg.norm(vector)

    if norm != 0:
        vector = vector / norm

    return vector

In [6]:
doc_embeddings = []

for doc in documents:
    emb = create_embedding(doc)
    doc_embeddings.append(emb)

doc_embeddings = np.array(doc_embeddings)

In [7]:
def retrieve(query, k=3):

    query_embedding = create_embedding(query)

    similarities = np.dot(doc_embeddings, query_embedding)

    top_k_idx = np.argsort(similarities)[-k:][::-1]

    results = [documents[i] for i in top_k_idx]

    return results

In [8]:
def build_prompt(query, contexts):

    context_text = "\n".join(contexts)

    prompt = f"""
You are a helpful assistant.

Answer the question using ONLY the context below.

Context:
{context_text}

Question:
{query}

Answer:
"""

    return prompt

In [12]:
def generate_answer(prompt):

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content

In [13]:
def rag_chat(query):

    contexts = retrieve(query)

    prompt = build_prompt(query, contexts)

    answer = generate_answer(prompt)

    return answer

In [17]:
query = "What is the capital of France?"

response = rag_chat(query)

print(response)

There is no information about the capital of France in the provided context. The context only discusses advances in artificial intelligence, natural language processing, and literature review processes, as well as a project about a dynamic research paper retrieval system.
